# 02 · Custom DataLoader for the fine-tuning strategy

Builds `TableImageJsonDataset` + `VLMCollator` **on top of the dataset from
notebook 01** and dissects exactly what a training batch contains: pixel
tensors, token ids, the `-100` label mask (loss only on the JSON target), and
the structure-vs-content **loss weight tags**.

> Requires the local Gemma folder (tokenizer only is touched here).

In [ ]:
# --- bootstrap: make the src/ package importable from notebooks/ ---
import sys, os
from pathlib import Path
REPO = Path.cwd().parent if (Path.cwd().name == "notebooks") else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)  # so relative paths in configs/config.yaml resolve

from gemma_ft_json.config import load_config
cfg = load_config("configs/config.yaml")
print("config loaded; data_root =", cfg.paths.data_root)


In [ ]:
from gemma_ft_json.models import load_gemma_local
from gemma_ft_json.utils import resolve_dtype
# Tokenizer comes from the LOCAL Gemma dir — offline mode is force-enabled.
_, tok = load_gemma_local(cfg.paths.gemma_model_dir, resolve_dtype(cfg.device.dtype))
print("vocab:", tok.vocab_size, "| pad id:", tok.pad_token_id)

## 1. Dataset (stage = `sft` → JSON targets)

In [ ]:
from gemma_ft_json.data import TableImageJsonDataset
ds = TableImageJsonDataset(cfg.paths.manifest_train, tok, cfg.dataset.image_size,
                           stage="sft", max_seq_len=cfg.model.max_seq_len)
print(len(ds), "samples")
sample = ds[0]
{k: tuple(v.shape) for k, v in sample.items()}

## 2. Decode one sample: prompt vs target vs label mask

In [ ]:
ids, labels = sample["input_ids"].tolist(), sample["labels"].tolist()
n_prompt = sum(1 for l in labels if l == -100)
print("PROMPT  :", repr(tok.decode(ids[:n_prompt])))
print("TARGET  :", tok.decode(ids[n_prompt:])[:200], "...")
print(f"label mask: {n_prompt} masked (-100) + {len(ids)-n_prompt} scored tokens")

## 3. Collated batch (dynamic padding) + weight-tag resolution

In [ ]:
from torch.utils.data import DataLoader
from gemma_ft_json.data import VLMCollator

coll = VLMCollator(tok.pad_token_id,
                   cfg.training.structure_loss_weight,
                   cfg.training.content_loss_weight)
loader = DataLoader(ds, batch_size=cfg.training.batch_size, shuffle=True,
                    collate_fn=coll, num_workers=0)
batch = next(iter(loader))
for k, v in batch.items():
    print(f"{k:16s} {tuple(v.shape)} {v.dtype}")

In [ ]:
# Verify weights: structure tokens (",{}:) carry 0.6, content carries 1.4
import torch
w, lab = batch["loss_weights"][0], batch["labels"][0]
scored = lab != -100
print("unique weights on scored tokens:", torch.unique(w[scored]).tolist())
print("share of content-weighted tokens:",
      f"{(w[scored] == cfg.training.content_loss_weight).float().mean():.1%}")

## 4. Why this matters
The collator's `loss_weights` make cross-entropy a faithful proxy for
**cell-level accuracy**: JSON scaffolding is learned in minutes and would
otherwise dominate (and flatter) the loss curve.